In [2]:
import torch
from torch import nn
import math

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        self.vocab_size = vocab_size

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [ ]:
d_model = 8
vocab_size = 1000
embedding_layer = InputEmbedding(vocab_size, d_model)
sentence_tokens = torch.tensor([[10, 25, 500, 30, 31, 85]])
output = embedding_layer(sentence_tokens)
print(output.shape)

torch.Size([1, 6, 8])


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, seq_len: int):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        self.seq_len = seq_len
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

In [ ]:
def forward(self, x):
        x = x + self.pe[:, :x.shape[1], :].requires_grad_(False)
        return self.dropout(x)

In [ ]:
from re import M

from regex import F
from torch import dropout


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, d_model: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(queries, keys, values, mask=None):
        d_k = queries.size(-1)
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float(-1e9))
        attention_probs = F.softmax(scores, dim=-1)
        if dropout is not None:
            attention_probs = dropout(attention_probs)
        return (scores @ values, attention_probs)
        attn_weights = torch.softmax(scores, dim=-1)
    def forward(self, v, k, q, mask):
        query = self.wq(q)
        key = self.wk(k)
        value = self.wv(v)

        query = query.view(query[0], query.shape[1], self.num_heads, self.d_k).transpose(1, 2)
        key = key.view(key[0], key.shape[1], self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(value[0], value.shape[1], self.num_heads, self.d_k).transpose(1, 2)

        x, self.scores = MultiHeadAttention.attention(query, key, value, mask)
        return self.w_o(x)